# 03 — Feature Engineering

Show every transformation applied to the raw data, explain the rationale, and visualise the sklearn preprocessing pipeline.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config

sns.set_theme(style="whitegrid")
set_config(display="diagram")

FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "train.csv")
print("Raw columns:", train.columns.tolist())

## 1. Raw Data Overview

In [ ]:
train[["Vehicle_Age", "Vehicle_Damage", "Gender"]].head(8)

## 2. Feature Transformations Applied

| Column | Raw Values | Transformation | Result |
|---|---|---|---|
| `Vehicle_Age` | `< 1 Year`, `1-2 Year`, `> 2 Years` | String normalisation → one-hot encoding | 3 binary columns |
| `Vehicle_Damage` | `Yes`, `No` | Map to 1/0 | Numeric binary |
| `Gender` | `Male`, `Female` | Map to 1/0 | Numeric binary |
| Numeric columns | raw floats | Median imputation → StandardScaler | Zero-mean, unit-std |
| `Vehicle_Age` (str) | normalised strings | OneHotEncoder | Sparse binary |

### Why StandardScaler?
Logistic Regression and distance-based models converge faster and produce more stable coefficients when features are on the same scale. Tree-based models (Random Forest, LightGBM) are scale-invariant, but keeping scaling in the pipeline ensures it works for all models without modification.

### Why OneHotEncoder for Vehicle_Age?
`Vehicle_Age` is **ordinal** (< 1 year < 1-2 years < > 2 years), but the relationship with the target is not necessarily linear. One-hot encoding lets the model learn each category independently. An ordinal encoder would impose a linear ordering assumption.

In [ ]:
from health_insurance_cross_sell.config import load_config
from health_insurance_cross_sell.features import prepare_features, model_matrix

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
df_prepared = prepare_features(train.copy(), config, training=True)

print("After feature engineering:")
df_prepared[["vehicle_age", "vehicle_damage", "gender"]].head(8)

In [ ]:
X, y, _ = model_matrix(df_prepared, config, training=True)
print(f"Feature matrix shape: {X.shape}")
print(f"\nColumn dtypes after prepare_features:")
X.dtypes

## 3. Sklearn Pipeline Diagram

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_cols = X.select_dtypes(exclude=["object", "category", "bool"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
])

full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

full_pipeline

In [ ]:
print(f"Numeric features  ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

## 4. Feature Distributions After Engineering

In [ ]:
key_numeric = [c for c in ["age", "annual_premium", "vintage"] if c in X.columns]

fig, axes = plt.subplots(1, len(key_numeric), figsize=(5 * len(key_numeric), 4))
if len(key_numeric) == 1:
    axes = [axes]

for ax, col in zip(axes, key_numeric):
    X_with_target = X.copy()
    X_with_target["Response"] = y.values
    for resp, color, label in [(0, "#5b8db8", "0"), (1, "#e07b39", "1")]:
        subset = X_with_target[X_with_target["Response"] == resp][col]
        subset.hist(bins=40, alpha=0.6, color=color, label=f"Response={label}", ax=ax)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle("Numeric Feature Distributions by Response", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "03_numeric_distributions.png", dpi=150)
plt.show()

## Summary

- `vehicle_damage` and `gender` are mapped to binary integers so they are treated as numeric by the pipeline.
- `vehicle_age` is string-normalised and fed into the categorical branch for one-hot encoding.
- All transformations are encapsulated in a single sklearn `Pipeline`, ensuring that test data goes through exactly the same transformations as training data — no leakage.